# ONNX for Generative AI — Deep Dive

This notebook explores the mathematical foundations of generative models—diffusion models
and autoregressive LLMs—and their deployment with ONNX Runtime. We cover the forward/reverse
diffusion process, KV-cache mechanics, and multi-component model export strategies.

## 1. Generative AI Landscape

```
┌──────────────────────────────────────────────────────────────────┐
│              GENERATIVE AI MODEL FAMILIES                          │
├──────────────────────────────────────────────────────────────────┤
│                                                                    │
│  ┌─────────────────────┐  ┌─────────────────────┐               │
│  │  DIFFUSION MODELS   │  │  AUTOREGRESSIVE LLMs │               │
│  │  (Stable Diffusion) │  │  (GPT, LLaMA)        │               │
│  ├─────────────────────┤  ├─────────────────────┤               │
│  │ • Text → Image      │  │ • Text → Text       │               │
│  │ • Image → Image     │  │ • Code generation   │               │
│  │ • Inpainting        │  │ • Chat/Dialog       │               │
│  │ • Super-resolution  │  │ • Summarization     │               │
│  └─────────────────────┘  └─────────────────────┘               │
│                                                                    │
│  ┌─────────────────────┐  ┌─────────────────────┐               │
│  │       GANs           │  │       VAEs           │               │
│  │  (StyleGAN, etc.)    │  │  (Image codec)       │               │
│  ├─────────────────────┤  ├─────────────────────┤               │
│  │ • Face generation   │  │ • Image compression │               │
│  │ • Style transfer    │  │ • Latent spaces     │               │
│  │ • Data augmentation │  │ • Anomaly detection │               │
│  └─────────────────────┘  └─────────────────────┘               │
│                                                                    │
│  ONNX Deployment Challenges:                                      │
│  • Multi-component pipelines (UNet + VAE + TextEncoder)          │
│  • Iterative inference (diffusion steps, autoregressive tokens)  │
│  • Large model sizes (billions of parameters)                    │
│  • Dynamic KV-cache growth                                       │
└──────────────────────────────────────────────────────────────────┘
```

## 2. Diffusion Models: Forward Process

Diffusion models learn to reverse a gradual noising process.

### Forward (Noising) Process

Given data $x_0 \sim q(x_0)$, we add Gaussian noise over $T$ steps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t I)$$

Where $\beta_t \in (0, 1)$ is the noise schedule (e.g., linear from $10^{-4}$ to $0.02$).

### Closed-Form Sampling

Define $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$. Then:

$$q(x_t | x_0) = \mathcal{N}(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) I)$$

This means we can sample any $x_t$ directly:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

```
Forward Process Visualization:

t=0          t=T/4         t=T/2         t=3T/4        t=T
┌────────┐   ┌────────┐   ┌────────┐   ┌────────┐   ┌────────┐
│ Clean  │──▶│ Slight │──▶│ Medium │──▶│ Heavy  │──▶│ Pure   │
│ Image  │   │ Noise  │   │ Noise  │   │ Noise  │   │ Noise  │
│ x₀     │   │        │   │        │   │        │   │ x_T~N  │
└────────┘   └────────┘   └────────┘   └────────┘   └────────┘
ᾱ_t ≈ 1.0   ᾱ_t ≈ 0.75  ᾱ_t ≈ 0.5   ᾱ_t ≈ 0.25  ᾱ_t ≈ 0.0
```

In [ ]:
import numpy as np

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    """Linear noise schedule."""
    return np.linspace(beta_start, beta_end, T)

def cosine_beta_schedule(T, s=0.008):
    """Cosine noise schedule (improved diffusion)."""
    steps = np.arange(T + 1) / T
    alphas_cumprod = np.cos((steps + s) / (1 + s) * np.pi / 2) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    return np.clip(betas, 0, 0.999)

T = 1000
betas_linear = linear_beta_schedule(T)
betas_cosine = cosine_beta_schedule(T)

# Compute alpha_bar for both schedules
alphas_linear = 1.0 - betas_linear
alpha_bar_linear = np.cumprod(alphas_linear)

alphas_cosine = 1.0 - betas_cosine
alpha_bar_cosine = np.cumprod(alphas_cosine)

print("Noise Schedule Comparison")
print("=" * 55)
print(f"{'Timestep':>10} {'ᾱ (linear)':>12} {'ᾱ (cosine)':>12} {'SNR_linear':>12}")
print("-" * 55)
for t in [0, 100, 250, 500, 750, 900, 999]:
    snr = alpha_bar_linear[t] / (1 - alpha_bar_linear[t])
    print(f"{t:>10} {alpha_bar_linear[t]:>12.6f} {alpha_bar_cosine[t]:>12.6f} {snr:>12.4f}")

print(f"\nSignal-to-Noise Ratio: SNR(t) = ᾱ_t / (1 - ᾱ_t)")
print(f"At t=0:   SNR ≈ {alpha_bar_linear[0]/(1-alpha_bar_linear[0]):.0f} (mostly signal)")
print(f"At t=999: SNR ≈ {alpha_bar_linear[999]/(1-alpha_bar_linear[999]):.6f} (mostly noise)")

## 3. Diffusion Models: Reverse Process

The reverse process learns to denoise step by step:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 I)$$

The mean is parameterized as:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t)\right)$$

Where $\epsilon_\theta$ is a neural network (UNet) that predicts the noise.

### Training Objective

The simplified loss (Ho et al., 2020):

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\left[\| \epsilon - \epsilon_\theta(x_t, t) \|^2\right]$$

This is just MSE between actual noise and predicted noise.

### DDPM Sampling Algorithm

```
Algorithm: DDPM Sampling
────────────────────────────────
1. x_T ~ N(0, I)
2. For t = T, T-1, ..., 1:
   a. z ~ N(0, I) if t > 1, else z = 0
   b. ε = ε_θ(x_t, t)        ← UNet forward pass
   c. x_{t-1} = (1/√α_t)(x_t - β_t/√(1-ᾱ_t) · ε) + σ_t · z
3. Return x_0
────────────────────────────────
Total UNet calls: T (e.g., 1000)
With DDIM: ~20-50 steps sufficient
```

In [ ]:
class DiffusionScheduler:
    """Implements DDPM and DDIM sampling schedules."""
    
    def __init__(self, T=1000, schedule='linear'):
        self.T = T
        
        if schedule == 'linear':
            self.betas = linear_beta_schedule(T)
        else:
            self.betas = cosine_beta_schedule(T)
        
        self.alphas = 1.0 - self.betas
        self.alpha_bar = np.cumprod(self.alphas)
        self.alpha_bar_prev = np.append(1.0, self.alpha_bar[:-1])
        
        # Posterior variance
        self.posterior_variance = self.betas * (1 - self.alpha_bar_prev) / (1 - self.alpha_bar)
    
    def add_noise(self, x0, t, noise=None):
        """Forward process: add noise to x0 at timestep t."""
        if noise is None:
            noise = np.random.randn(*x0.shape)
        
        sqrt_alpha_bar = np.sqrt(self.alpha_bar[t])
        sqrt_one_minus_alpha_bar = np.sqrt(1 - self.alpha_bar[t])
        
        return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise
    
    def ddpm_step(self, x_t, predicted_noise, t):
        """Single DDPM reverse step."""
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alpha_bar[t]
        beta_t = self.betas[t]
        
        # Compute mean
        mean = (1 / np.sqrt(alpha_t)) * (
            x_t - (beta_t / np.sqrt(1 - alpha_bar_t)) * predicted_noise
        )
        
        # Add noise (except at t=0)
        if t > 0:
            noise = np.random.randn(*x_t.shape)
            sigma = np.sqrt(self.posterior_variance[t])
            return mean + sigma * noise
        return mean
    
    def ddim_step(self, x_t, predicted_noise, t, t_prev, eta=0.0):
        """Single DDIM step (deterministic when eta=0)."""
        alpha_bar_t = self.alpha_bar[t]
        alpha_bar_prev = self.alpha_bar[t_prev] if t_prev >= 0 else 1.0
        
        # Predict x_0
        x0_pred = (x_t - np.sqrt(1 - alpha_bar_t) * predicted_noise) / np.sqrt(alpha_bar_t)
        
        # Direction pointing to x_t
        sigma = eta * np.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev))
        dir_xt = np.sqrt(1 - alpha_bar_prev - sigma**2) * predicted_noise
        
        x_prev = np.sqrt(alpha_bar_prev) * x0_pred + dir_xt
        if eta > 0 and t_prev > 0:
            x_prev += sigma * np.random.randn(*x_t.shape)
        
        return x_prev

# Demonstrate forward process
scheduler = DiffusionScheduler(T=1000)
x0 = np.random.randn(1, 4, 64, 64).astype(np.float32)  # Latent

print("Forward Diffusion Process:")
print("=" * 50)
for t in [0, 100, 250, 500, 750, 999]:
    x_t = scheduler.add_noise(x0, t)
    signal_power = np.var(np.sqrt(scheduler.alpha_bar[t]) * x0)
    noise_power = np.var(np.sqrt(1 - scheduler.alpha_bar[t]) * np.random.randn(*x0.shape))
    print(f"  t={t:4d}: std={np.std(x_t):.4f}, signal={signal_power:.4f}, noise={noise_power:.4f}")

## 4. UNet Architecture for Diffusion

The UNet predicts noise $\epsilon_\theta(x_t, t)$ conditioned on timestep and text:

```
┌───────────────────────────────────────────────────────────────────┐
│                  STABLE DIFFUSION UNet                              │
├───────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Inputs:                                                            │
│    • x_t ∈ R^(B×4×64×64)  — noisy latent                          │
│    • t ∈ R^B              — timestep embedding                     │
│    • c ∈ R^(B×77×768)    — text conditioning (CLIP)               │
│                                                                     │
│  ┌──────────────────────────────────────────────────────┐         │
│  │              ENCODER (Down-sampling)                    │         │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐           │         │
│  │  │ ResBlock  │  │ ResBlock  │  │ ResBlock  │           │         │
│  │  │ + CrossAtt│──▶│ + CrossAtt│──▶│ + CrossAtt│           │         │
│  │  │ 320ch     │  │ 640ch     │  │ 1280ch    │           │         │
│  │  └──────┬───┘  └──────┬───┘  └──────┬───┘           │         │
│  │         │↓2×          │↓2×          │↓2×             │         │
│  └─────────┼──────────────┼──────────────┼──────────────┘         │
│            │              │              │                          │
│  ┌─────────┼──────────────┼──────────────┼──────────────┐         │
│  │         │    MIDDLE BLOCK              │              │         │
│  │         │    (ResBlock + CrossAttn + ResBlock)         │         │
│  └─────────┼──────────────┼──────────────┼──────────────┘         │
│            │              │              │                          │
│  ┌─────────┼──────────────┼──────────────┼──────────────┐         │
│  │         │↑2×          │↑2×          │↑2×             │         │
│  │  ┌──────┴───┐  ┌──────┴───┐  ┌──────┴───┐           │         │
│  │  │ ResBlock  │  │ ResBlock  │  │ ResBlock  │           │         │
│  │  │ + CrossAtt│◀─│ + CrossAtt│◀─│ + CrossAtt│           │         │
│  │  │ + Skip    │  │ + Skip    │  │ + Skip    │           │         │
│  │  └──────────┘  └──────────┘  └──────────┘           │         │
│  │              DECODER (Up-sampling)                     │         │
│  └──────────────────────────────────────────────────────┘         │
│                                                                     │
│  Output: ε_θ ∈ R^(B×4×64×64)  — predicted noise                   │
└───────────────────────────────────────────────────────────────────┘
```

### Cross-Attention for Text Conditioning

$$\text{CrossAttn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$$

Where $Q$ comes from the image features and $K, V$ from text embeddings.

In [ ]:
import torch
import torch.nn as nn

class TimeEmbedding(nn.Module):
    """Sinusoidal timestep embedding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim)
        )
    
    def forward(self, t):
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float()[:, None] * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return self.mlp(emb)

class CrossAttention(nn.Module):
    """Cross-attention for text conditioning."""
    def __init__(self, dim, context_dim=768, heads=8):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(context_dim, dim)
        self.to_v = nn.Linear(context_dim, dim)
        self.to_out = nn.Linear(dim, dim)
    
    def forward(self, x, context):
        B, N, D = x.shape
        q = self.to_q(x).view(B, N, self.heads, self.head_dim).transpose(1, 2)
        k = self.to_k(context).view(B, -1, self.heads, self.head_dim).transpose(1, 2)
        v = self.to_v(context).view(B, -1, self.heads, self.head_dim).transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.to_out(out)

class SimplifiedUNetBlock(nn.Module):
    """Single UNet block with ResNet + CrossAttention."""
    def __init__(self, channels, time_dim=256, context_dim=768):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, channels)
        self.time_proj = nn.Linear(time_dim, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, channels)
        self.cross_attn = CrossAttention(channels, context_dim, heads=4)
        self.attn_norm = nn.LayerNorm(channels)
    
    def forward(self, x, t_emb, context):
        B, C, H, W = x.shape
        # ResBlock
        h = torch.silu(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = torch.silu(self.norm2(self.conv2(h)))
        h = h + x  # Residual
        
        # Cross-Attention
        h_flat = h.reshape(B, C, H*W).transpose(1, 2)  # [B, H*W, C]
        h_flat = self.attn_norm(h_flat)
        h_flat = h_flat + self.cross_attn(h_flat, context)
        h = h_flat.transpose(1, 2).reshape(B, C, H, W)
        
        return h

# Test
block = SimplifiedUNetBlock(channels=64)
x = torch.randn(1, 64, 32, 32)
t_emb = torch.randn(1, 256)
context = torch.randn(1, 77, 768)

with torch.no_grad():
    out = block(x, t_emb, context)
print(f"UNet Block: input={list(x.shape)}, t_emb={list(t_emb.shape)}, "
      f"context={list(context.shape)} → output={list(out.shape)}")

## 5. Stable Diffusion Pipeline Components

Stable Diffusion consists of multiple separately-exportable models:

```
┌──────────────────────────────────────────────────────────────┐
│           STABLE DIFFUSION PIPELINE                           │
├──────────────────────────────────────────────────────────────┤
│                                                                │
│  "A photo of a cat"                                           │
│       │                                                        │
│       ▼                                                        │
│  ┌─────────────────────┐                                     │
│  │  TEXT ENCODER (CLIP) │  → text_embeddings [1,77,768]      │
│  │  ~123M params        │     Export: text_encoder.onnx       │
│  └──────────┬──────────┘                                     │
│             │                                                  │
│             ▼                                                  │
│  ┌─────────────────────┐                                     │
│  │       UNet           │  → predicted noise [1,4,64,64]     │
│  │  ~860M params        │     Export: unet.onnx              │
│  │  Called T times      │     Inputs: latent, timestep, text │
│  └──────────┬──────────┘                                     │
│             │  (iterated T times)                             │
│             ▼                                                  │
│  Denoised latent [1,4,64,64]                                 │
│             │                                                  │
│             ▼                                                  │
│  ┌─────────────────────┐                                     │
│  │  VAE DECODER         │  → image [1,3,512,512]             │
│  │  ~50M params         │     Export: vae_decoder.onnx       │
│  └─────────────────────┘                                     │
│                                                                │
│  Optional: Safety Checker (~155M params)                      │
└──────────────────────────────────────────────────────────────┘
```

### Memory Requirements (FP16)

| Component | Parameters | FP16 Size | Calls per Image |
|-----------|-----------|-----------|------------------|
| Text Encoder | 123M | 246 MB | 1 |
| UNet | 860M | 1.7 GB | 20-50 (steps) |
| VAE Decoder | 50M | 100 MB | 1 |
| **Total** | **~1B** | **~2 GB** | - |

In [ ]:
# Memory calculator for diffusion models
def diffusion_memory_estimate(batch_size=1, num_steps=50, latent_size=64,
                              channels=4, precision='fp16'):
    """Estimate memory requirements for diffusion inference."""
    bytes_per_element = 2 if precision == 'fp16' else 4
    
    # Model weights
    text_encoder_params = 123e6
    unet_params = 860e6
    vae_params = 50e6
    
    model_memory = (text_encoder_params + unet_params + vae_params) * bytes_per_element
    
    # Activation memory (during UNet forward)
    latent_size_bytes = batch_size * channels * latent_size * latent_size * bytes_per_element
    
    # UNet intermediate activations (rough estimate)
    unet_activations = latent_size_bytes * 50  # ~50x latent size for intermediates
    
    # Text embeddings
    text_memory = batch_size * 77 * 768 * bytes_per_element
    
    total = model_memory + unet_activations + text_memory
    
    return {
        'model_weights_gb': model_memory / 1e9,
        'activations_gb': unet_activations / 1e9,
        'latent_mb': latent_size_bytes / 1e6,
        'total_gb': total / 1e9,
        'total_unet_calls': num_steps * (2 if batch_size > 0 else 1),  # CFG doubles calls
    }

print("Stable Diffusion Memory Analysis")
print("=" * 55)
for precision in ['fp32', 'fp16']:
    mem = diffusion_memory_estimate(precision=precision)
    print(f"\n  {precision.upper()}:")
    print(f"    Model weights: {mem['model_weights_gb']:.2f} GB")
    print(f"    Activations:   {mem['activations_gb']:.2f} GB")
    print(f"    Total:         {mem['total_gb']:.2f} GB")
    print(f"    UNet calls:    {mem['total_unet_calls']} (with CFG)")

## 6. DDIM Sampling (Accelerated)

DDIM (Denoising Diffusion Implicit Models) allows fewer steps:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot \underbrace{\frac{x_t - \sqrt{1-\bar{\alpha}_t} \cdot \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}}_{\text{predicted } x_0} + \sqrt{1-\bar{\alpha}_{t-1}-\sigma_t^2} \cdot \epsilon_\theta + \sigma_t \cdot \epsilon$$

When $\sigma_t = 0$: deterministic (DDIM)
When $\sigma_t = \sqrt{\beta_t}$: stochastic (DDPM)

### Step Reduction

| Method | Steps | Quality | Speed |
|--------|-------|---------|-------|
| DDPM | 1000 | Best | Very slow |
| DDIM | 50 | Good | 20× faster |
| DDIM | 20 | Acceptable | 50× faster |
| DPM-Solver++ | 20 | Good | 50× faster |
| LCM | 4-8 | Good | 125-250× faster |

In [ ]:
def simulate_diffusion_sampling(scheduler, unet_fn, shape, num_steps=50,
                                 method='ddim', eta=0.0):
    """Simulate diffusion sampling with a fake UNet."""
    # Create timestep schedule (evenly spaced)
    timesteps = np.linspace(scheduler.T - 1, 0, num_steps).astype(int)
    
    # Start from pure noise
    x = np.random.randn(*shape).astype(np.float32)
    
    trajectory = [('T', np.std(x))]
    
    for i in range(len(timesteps)):
        t = timesteps[i]
        t_prev = timesteps[i + 1] if i < len(timesteps) - 1 else 0
        
        # UNet predicts noise
        predicted_noise = unet_fn(x, t)
        
        if method == 'ddpm':
            x = scheduler.ddpm_step(x, predicted_noise, t)
        elif method == 'ddim':
            x = scheduler.ddim_step(x, predicted_noise, t, t_prev, eta=eta)
        
        if i % (num_steps // 5) == 0:
            trajectory.append((f't={t}', np.std(x)))
    
    trajectory.append(('Final', np.std(x)))
    return x, trajectory

# Fake UNet that just returns random noise (for demonstration)
def fake_unet(x, t):
    return np.random.randn(*x.shape).astype(np.float32) * 0.5

# Run sampling
shape = (1, 4, 32, 32)
x_final, traj = simulate_diffusion_sampling(scheduler, fake_unet, shape, num_steps=20)

print("DDIM Sampling Trajectory (20 steps):")
print("=" * 40)
for step, std in traj:
    bar = '█' * int(std * 10)
    print(f"  {step:>8}: std={std:.4f} {bar}")

print(f"\nFinal output shape: {x_final.shape}")
print(f"(In practice, this latent is decoded by VAE to produce an image)")

## 7. Classifier-Free Guidance (CFG)

CFG interpolates between conditional and unconditional predictions:

$$\tilde{\epsilon}_\theta(x_t, t, c) = \epsilon_\theta(x_t, t, \emptyset) + w \cdot (\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \emptyset))$$

Where:
- $c$ = text conditioning
- $\emptyset$ = unconditional (empty text)
- $w$ = guidance scale (typically 7.5)

This requires **two UNet forward passes per step** (or one batched pass with batch_size=2).

### ONNX Implications

```
With CFG (guidance_scale > 1):
  UNet input: [2, 4, 64, 64]     (batched: conditional + unconditional)
  UNet output: [2, 4, 64, 64]
  Split → apply guidance formula → [1, 4, 64, 64]

Without CFG (guidance_scale = 1):
  UNet input: [1, 4, 64, 64]
  UNet output: [1, 4, 64, 64]
```

In [ ]:
def classifier_free_guidance(noise_pred_cond, noise_pred_uncond, guidance_scale=7.5):
    """Apply classifier-free guidance."""
    return noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

# Demonstrate CFG effect
latent_shape = (1, 4, 64, 64)
noise_cond = np.random.randn(*latent_shape).astype(np.float32)
noise_uncond = np.random.randn(*latent_shape).astype(np.float32)

print("Classifier-Free Guidance Effect:")
print("=" * 55)
print(f"{'Scale':>8} {'Std(output)':>12} {'Max(output)':>12} {'Characteristic':>16}")
print("-" * 55)

for scale in [1.0, 3.0, 7.5, 12.0, 20.0]:
    guided = classifier_free_guidance(noise_cond, noise_uncond, scale)
    char = 'no guidance' if scale == 1 else 'weak' if scale < 5 else 'standard' if scale < 10 else 'strong' if scale < 15 else 'very strong'
    print(f"{scale:>8.1f} {np.std(guided):>12.4f} {np.max(np.abs(guided)):>12.4f} {char:>16}")

print("\nHigher guidance → more adherence to text prompt, less diversity")

## 8. KV-Cache for Autoregressive LLMs

For autoregressive generation, we cache Key/Value tensors to avoid recomputation:

### Memory Formula

$$\text{KV-Cache Memory} = 2 \cdot n_{layers} \cdot n_{heads} \cdot d_{head} \cdot \text{seq\_len} \cdot \text{batch} \cdot \text{bytes}$$

The factor of 2 accounts for both Keys and Values.

```
┌──────────────────────────────────────────────────────────────┐
│              KV-CACHE MECHANISM                                │
├──────────────────────────────────────────────────────────────┤
│                                                                │
│  Without KV-Cache (naive):                                    │
│  Step 1: Process [t₁]           → 1 token  computation       │
│  Step 2: Process [t₁, t₂]       → 2 tokens computation       │
│  Step 3: Process [t₁, t₂, t₃]   → 3 tokens computation       │
│  Step n: Process all n tokens    → n tokens computation       │
│  Total: 1+2+3+...+n = O(n²)                                  │
│                                                                │
│  With KV-Cache:                                               │
│  Step 1: Process [t₁]           → cache K₁,V₁                │
│  Step 2: Process [t₂] only      → cache K₂,V₂, attend to K₁₂│
│  Step 3: Process [t₃] only      → cache K₃,V₃, attend to K₁₂₃│
│  Step n: Process [tₙ] only      → 1 token computation        │
│  Total: n × O(1 new + attend to cache) = O(n)                │
│                                                                │
│  Speedup: O(n²) → O(n) per generation                        │
│  Trade-off: Uses more memory for cache storage                │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
def kv_cache_memory(n_layers, n_heads, d_head, seq_len, batch_size=1, 
                    dtype_bytes=2):
    """Calculate KV-cache memory in bytes."""
    # 2 for K and V, per layer
    return 2 * n_layers * n_heads * d_head * seq_len * batch_size * dtype_bytes

# Common model configurations
models = {
    'GPT-2 (124M)': (12, 12, 64, 1024),
    'GPT-2 XL (1.5B)': (48, 25, 64, 1024),
    'LLaMA-7B': (32, 32, 128, 2048),
    'LLaMA-13B': (40, 40, 128, 2048),
    'LLaMA-70B': (80, 64, 128, 4096),
}

print("KV-Cache Memory Analysis (FP16, batch=1)")
print("=" * 65)
print(f"{'Model':<18} {'Layers':>7} {'Heads':>6} {'d_head':>7} {'SeqLen':>7} {'KV-Cache':>10}")
print("-" * 65)

for name, (layers, heads, d_head, max_seq) in models.items():
    mem = kv_cache_memory(layers, heads, d_head, max_seq, dtype_bytes=2)
    if mem > 1e9:
        mem_str = f"{mem/1e9:.2f} GB"
    else:
        mem_str = f"{mem/1e6:.1f} MB"
    print(f"{name:<18} {layers:>7} {heads:>6} {d_head:>7} {max_seq:>7} {mem_str:>10}")

# Batch size scaling
print(f"\nKV-Cache scaling with batch size (LLaMA-7B, seq=2048):")
for bs in [1, 4, 8, 16, 32]:
    mem = kv_cache_memory(32, 32, 128, 2048, batch_size=bs, dtype_bytes=2)
    print(f"  batch={bs:>2}: {mem/1e9:.2f} GB")

## 9. ONNX Export for Diffusion Components

Each component is exported separately for maximum flexibility:

$$\text{Pipeline} = \text{TextEncoder}(\text{text}) \rightarrow \text{UNet}^{(T)}(z_t, t, c) \rightarrow \text{VAE}(z_0)$$

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

# Simplified components for export demo
class SimpleTextEncoder(nn.Module):
    def __init__(self, vocab_size=49408, dim=768, max_len=77):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)
        self.layers = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dim, 8, dim*4, batch_first=True), num_layers=2)
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, input_ids):
        seq_len = input_ids.shape[1]
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embed(input_ids) + self.pos(pos_ids)
        x = self.layers(x)
        return self.norm(x)

class SimpleVAEDecoder(nn.Module):
    def __init__(self, latent_ch=4, out_ch=3):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_ch, 128, 3, padding=1),
            nn.SiLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.SiLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.SiLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(32, out_ch, 3, padding=1),
            nn.Tanh()
        )
    
    def forward(self, latent):
        return self.decoder(latent)

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=4, time_dim=256, context_dim=768):
        super().__init__()
        self.time_embed = TimeEmbedding(time_dim)
        self.block1 = SimplifiedUNetBlock(64, time_dim, context_dim)
        self.block2 = SimplifiedUNetBlock(64, time_dim, context_dim)
        self.in_conv = nn.Conv2d(in_ch, 64, 3, padding=1)
        self.out_conv = nn.Conv2d(64, in_ch, 3, padding=1)
    
    def forward(self, x, timestep, context):
        t_emb = self.time_embed(timestep)
        h = self.in_conv(x)
        h = self.block1(h, t_emb, context)
        h = self.block2(h, t_emb, context)
        return self.out_conv(h)

# Export all components
text_encoder = SimpleTextEncoder()
text_encoder.eval()
unet = SimpleUNet()
unet.eval()
vae_decoder = SimpleVAEDecoder()
vae_decoder.eval()

# Export text encoder
torch.onnx.export(
    text_encoder, torch.randint(0, 49408, (1, 77)),
    'outputs/text_encoder.onnx',
    input_names=['input_ids'], output_names=['text_embeddings'],
    dynamic_axes={'input_ids': {0: 'batch'}, 'text_embeddings': {0: 'batch'}},
    opset_version=14
)

# Export UNet
torch.onnx.export(
    unet, (torch.randn(1,4,32,32), torch.tensor([500]), torch.randn(1,77,768)),
    'outputs/unet.onnx',
    input_names=['latent', 'timestep', 'context'],
    output_names=['noise_pred'],
    dynamic_axes={'latent': {0:'batch'}, 'context': {0:'batch'}, 'noise_pred': {0:'batch'}},
    opset_version=14
)

# Export VAE decoder
torch.onnx.export(
    vae_decoder, torch.randn(1,4,32,32),
    'outputs/vae_decoder.onnx',
    input_names=['latent'], output_names=['image'],
    dynamic_axes={'latent': {0:'batch'}, 'image': {0:'batch'}},
    opset_version=14
)

print("Exported Diffusion Components:")
for f in ['text_encoder.onnx', 'unet.onnx', 'vae_decoder.onnx']:
    size = os.path.getsize(f'outputs/{f}') / 1024 / 1024
    print(f"  {f:<25}: {size:.2f} MB")

## 10. Variational Autoencoder (VAE)

The VAE maps between pixel space and latent space:

### Encoder
$$z = \text{Encoder}(x) \in \mathbb{R}^{B \times 4 \times H/8 \times W/8}$$

### Decoder
$$\hat{x} = \text{Decoder}(z) \in \mathbb{R}^{B \times 3 \times H \times W}$$

### Latent Space Scaling
$$z_{\text{scaled}} = \frac{z}{\sigma_{\text{latent}}}$$

In Stable Diffusion, $\sigma_{\text{latent}} = 0.18215$ (learned from training data).

### Benefits of Latent Diffusion

| Space | Resolution | Channels | Pixels | Compute |
|-------|-----------|----------|--------|----------|
| Pixel (512×512) | 512×512 | 3 | 786,432 | Baseline |
| Latent (64×64) | 64×64 | 4 | 16,384 | ~48× less |

In [ ]:
import onnxruntime as ort

# Test the exported VAE decoder
vae_session = ort.InferenceSession('outputs/vae_decoder.onnx')

# Generate a random latent
latent = np.random.randn(1, 4, 32, 32).astype(np.float32)

# Decode
image = vae_session.run(None, {'latent': latent})[0]

print(f"VAE Decoder Test:")
print(f"  Latent input:  {latent.shape} ({latent.size * 4 / 1024:.1f} KB)")
print(f"  Image output:  {image.shape} ({image.size * 4 / 1024:.1f} KB)")
print(f"  Spatial upscale: {image.shape[2] // latent.shape[2]}×")
print(f"  Value range: [{image.min():.3f}, {image.max():.3f}] (tanh output)")
print(f"  Pixel space = (output + 1) / 2 × 255")

## 11. Complete Diffusion Pipeline with ONNX Runtime

Orchestrating all components together:

```
ONNX Diffusion Pipeline:
═══════════════════════════════════════════
1. Encode text      → text_encoder.onnx
2. Initialize noise → random z_T
3. For t in schedule:
   a. Concatenate z_t for CFG  (batch=2)
   b. Run UNet      → unet.onnx
   c. Apply CFG formula
   d. Scheduler step (DDIM/DPM++)
4. Decode latent    → vae_decoder.onnx
5. Post-process     → uint8 image
═══════════════════════════════════════════
```

In [ ]:
class ONNXDiffusionPipeline:
    """Complete diffusion pipeline using ONNX Runtime."""
    
    def __init__(self, text_encoder_path, unet_path, vae_path,
                 num_steps=20, guidance_scale=7.5):
        self.text_session = ort.InferenceSession(text_encoder_path)
        self.unet_session = ort.InferenceSession(unet_path)
        self.vae_session = ort.InferenceSession(vae_path)
        
        self.num_steps = num_steps
        self.guidance_scale = guidance_scale
        self.scheduler = DiffusionScheduler(T=1000)
    
    def encode_text(self, token_ids):
        """Encode text tokens to embeddings."""
        return self.text_session.run(None, {'input_ids': token_ids})[0]
    
    def denoise_step(self, latent, timestep, text_embeddings):
        """Single denoising step with CFG."""
        # Run UNet with conditional embedding
        noise_pred = self.unet_session.run(None, {
            'latent': latent,
            'timestep': np.array([timestep], dtype=np.int64),
            'context': text_embeddings
        })[0]
        
        # Run UNet with unconditional (zeros)
        uncond_embeddings = np.zeros_like(text_embeddings)
        noise_uncond = self.unet_session.run(None, {
            'latent': latent,
            'timestep': np.array([timestep], dtype=np.int64),
            'context': uncond_embeddings
        })[0]
        
        # Apply CFG
        noise_guided = classifier_free_guidance(
            noise_pred, noise_uncond, self.guidance_scale)
        
        return noise_guided
    
    def generate(self, token_ids, latent_shape=(1, 4, 32, 32), seed=42):
        """Generate an image from text tokens."""
        np.random.seed(seed)
        
        # 1. Encode text
        text_emb = self.encode_text(token_ids)
        
        # 2. Initialize noise
        latent = np.random.randn(*latent_shape).astype(np.float32)
        
        # 3. Denoise iteratively
        timesteps = np.linspace(999, 0, self.num_steps).astype(int)
        
        for i, t in enumerate(timesteps):
            noise_pred = self.denoise_step(latent, t, text_emb)
            t_prev = timesteps[i+1] if i < len(timesteps)-1 else 0
            latent = self.scheduler.ddim_step(latent, noise_pred, t, t_prev)
        
        # 4. Decode latent to image
        image = self.vae_session.run(None, {'latent': latent.astype(np.float32)})[0]
        
        # 5. Post-process
        image = ((image + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
        
        return image

# Run the pipeline
pipeline = ONNXDiffusionPipeline(
    'outputs/text_encoder.onnx',
    'outputs/unet.onnx',
    'outputs/vae_decoder.onnx',
    num_steps=10,
    guidance_scale=7.5
)

# Simulate token IDs
fake_tokens = np.random.randint(0, 49408, (1, 77)).astype(np.int64)

import time
start = time.perf_counter()
result = pipeline.generate(fake_tokens)
elapsed = time.perf_counter() - start

print(f"Diffusion Pipeline Results:")
print(f"  Steps: 10")
print(f"  Guidance scale: 7.5")
print(f"  Output shape: {result.shape}")
print(f"  Time: {elapsed:.2f}s ({elapsed/10*1000:.0f}ms per step)")
print(f"  Value range: [{result.min()}, {result.max()}]")

## 12. LLM Inference with KV-Cache in ONNX

Autoregressive generation with ONNX requires passing KV-cache as model I/O:

```
ONNX Model I/O for LLM with KV-Cache:
───────────────────────────────────────────
Inputs:
  • input_ids:    [batch, 1]              (single new token)
  • past_key_0:   [batch, heads, past_seq, d_head]
  • past_value_0: [batch, heads, past_seq, d_head]
  • past_key_1:   [batch, heads, past_seq, d_head]
  • past_value_1: ...
  ... (for each layer)

Outputs:
  • logits:        [batch, 1, vocab_size]
  • present_key_0: [batch, heads, past_seq+1, d_head]
  • present_value_0: [batch, heads, past_seq+1, d_head]
  ... (updated cache for each layer)
───────────────────────────────────────────
```

In [ ]:
class LLMWithKVCache(nn.Module):
    """Simple LLM demonstrating KV-cache export pattern."""
    
    def __init__(self, vocab_size=32000, dim=256, num_heads=8, num_layers=2):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.num_layers = num_layers
        
        self.embed = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            nn.MultiheadAttention(dim, num_heads, batch_first=True)
            for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(num_layers)])
        self.head = nn.Linear(dim, vocab_size)
    
    def forward(self, input_ids, past_key_values=None):
        x = self.embed(input_ids)
        
        new_key_values = []
        for i, (attn, norm) in enumerate(zip(self.layers, self.norms)):
            residual = x
            x = norm(x)
            
            # In a full implementation, we'd properly handle KV-cache here
            attn_out, _ = attn(x, x, x)
            x = residual + attn_out
        
        logits = self.head(x)
        return logits

llm = LLMWithKVCache(vocab_size=32000, dim=256, num_heads=8, num_layers=2)
llm.eval()

# Export without KV-cache (simpler version)
llm_path = 'outputs/llm_model.onnx'
dummy_ids = torch.randint(0, 32000, (1, 16))

torch.onnx.export(
    llm, dummy_ids, llm_path,
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'logits': {0: 'batch', 1: 'seq'}},
    opset_version=14
)

print(f"LLM exported: {os.path.getsize(llm_path)/1024/1024:.2f} MB")
print(f"Parameters: {sum(p.numel() for p in llm.parameters()):,}")

## 13. Autoregressive Generation with ONNX

Generate text token by token:

$$P(x_{1:T}) = \prod_{t=1}^{T} P(x_t | x_{1:t-1})$$

In [ ]:
class ONNXLLMGenerator:
    """Autoregressive text generation with ONNX Runtime."""
    
    def __init__(self, model_path):
        self.session = ort.InferenceSession(model_path)
    
    def generate(self, prompt_ids, max_new_tokens=50, temperature=1.0, top_k=50):
        """Generate tokens autoregressively."""
        generated = list(prompt_ids[0])
        
        for step in range(max_new_tokens):
            # Without KV-cache: reprocess entire sequence
            input_ids = np.array([generated], dtype=np.int64)
            logits = self.session.run(None, {'input_ids': input_ids})[0]
            
            # Get logits for last position
            next_logits = logits[0, -1, :] / temperature
            
            # Top-k sampling
            top_k_idx = np.argpartition(next_logits, -top_k)[-top_k:]
            top_k_logits = next_logits[top_k_idx]
            probs = np.exp(top_k_logits - top_k_logits.max())
            probs /= probs.sum()
            
            next_token = top_k_idx[np.random.choice(len(top_k_idx), p=probs)]
            generated.append(int(next_token))
        
        return np.array([generated])

# Generate
generator = ONNXLLMGenerator('outputs/llm_model.onnx')
prompt = np.array([[1, 500, 1000, 1500]], dtype=np.int64)

start = time.perf_counter()
output = generator.generate(prompt, max_new_tokens=20, temperature=0.8)
elapsed = (time.perf_counter() - start) * 1000

print(f"LLM Generation:")
print(f"  Prompt length: {len(prompt[0])} tokens")
print(f"  Generated: {len(output[0]) - len(prompt[0])} new tokens")
print(f"  Total time: {elapsed:.1f} ms")
print(f"  Per token: {elapsed / 20:.1f} ms")
print(f"  Tokens/sec: {20 / (elapsed/1000):.1f}")

## 14. Optimization Strategies for Generative Models

```
┌──────────────────────────────────────────────────────────────┐
│         OPTIMIZATION STRATEGIES                                │
├──────────────────────────────────────────────────────────────┤
│                                                                │
│  1. PRECISION REDUCTION                                       │
│     FP32 → FP16: 2× memory saving, ~1.5× speedup            │
│     FP16 → INT8: 2× memory saving (weights only for LLMs)   │
│     INT4 (GPTQ/AWQ): 4× memory savings for LLMs             │
│                                                                │
│  2. FEWER STEPS (Diffusion)                                   │
│     1000 → 50 (DDIM): 20× speedup                           │
│     50 → 4 (LCM/Turbo): 12.5× additional                    │
│                                                                │
│  3. MODEL DISTILLATION                                        │
│     SD XL → SD Turbo: similar quality, 1 step                │
│     70B → 7B: knowledge distillation                          │
│                                                                │
│  4. GRAPH OPTIMIZATION                                        │
│     Attention fusion (Flash Attention pattern)                │
│     Conv+BN fusion                                            │
│     Constant folding                                          │
│                                                                │
│  5. SPECULATIVE DECODING (LLMs)                               │
│     Draft model generates candidates                          │
│     Target model verifies in parallel                         │
│     ~2-3× speedup without quality loss                        │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Quantize the UNet
unet_int8_path = 'outputs/unet_int8.onnx'
quantize_dynamic('outputs/unet.onnx', unet_int8_path, weight_type=QuantType.QInt8)

# Quantize LLM
llm_int8_path = 'outputs/llm_int8.onnx'
quantize_dynamic('outputs/llm_model.onnx', llm_int8_path, weight_type=QuantType.QInt8)

# Compare sizes and speeds
print("Quantization Results for Generative Models")
print("=" * 55)
print(f"{'Component':<20} {'FP32 (MB)':>10} {'INT8 (MB)':>10} {'Ratio':>8}")
print("-" * 55)

for name, fp32, int8 in [
    ('UNet', 'outputs/unet.onnx', unet_int8_path),
    ('LLM', 'outputs/llm_model.onnx', llm_int8_path),
    ('Text Encoder', 'outputs/text_encoder.onnx', None),
    ('VAE Decoder', 'outputs/vae_decoder.onnx', None),
]:
    s1 = os.path.getsize(fp32) / 1024 / 1024
    if int8:
        s2 = os.path.getsize(int8) / 1024 / 1024
        print(f"{name:<20} {s1:>10.2f} {s2:>10.2f} {s1/s2:>7.2f}×")
    else:
        print(f"{name:<20} {s1:>10.2f} {'N/A':>10} {'':>8}")

# Speed comparison for UNet
unet_fp32 = ort.InferenceSession('outputs/unet.onnx')
unet_int8 = ort.InferenceSession(unet_int8_path)

test_data = {
    'latent': np.random.randn(1,4,32,32).astype(np.float32),
    'timestep': np.array([500], dtype=np.int64),
    'context': np.random.randn(1,77,768).astype(np.float32)
}

# Benchmark
for _ in range(5):
    unet_fp32.run(None, test_data)
    unet_int8.run(None, test_data)

times_fp32 = []
times_int8 = []
for _ in range(20):
    t0 = time.perf_counter()
    unet_fp32.run(None, test_data)
    times_fp32.append((time.perf_counter()-t0)*1000)
    
    t0 = time.perf_counter()
    unet_int8.run(None, test_data)
    times_int8.append((time.perf_counter()-t0)*1000)

print(f"\nUNet Speed: FP32={np.mean(times_fp32):.2f}ms, INT8={np.mean(times_int8):.2f}ms")
print(f"Speedup: {np.mean(times_fp32)/np.mean(times_int8):.2f}×")

## 15. Latent Consistency Models (LCM)

LCM reduces diffusion steps from 50 to 4-8 through consistency distillation:

$$f_\theta(x_t, t) \approx x_0 \quad \forall t$$

The consistency property ensures the model maps any noisy version
of the same image to the same clean output, regardless of noise level.

### Inference Speed Comparison

| Method | Steps | Time (A100) | Time (CPU) |
|--------|-------|-------------|------------|
| DDPM | 1000 | ~30s | ~10min |
| DDIM | 50 | ~1.5s | ~30s |
| DPM++ | 20 | ~0.6s | ~12s |
| LCM | 4 | ~0.12s | ~2.5s |
| SD Turbo | 1 | ~0.03s | ~0.6s |

In [ ]:
# Simulate generation time scaling with steps
import time

unet_session = ort.InferenceSession('outputs/unet.onnx')

# Measure single UNet call
single_call_times = []
for _ in range(20):
    t0 = time.perf_counter()
    unet_session.run(None, test_data)
    single_call_times.append((time.perf_counter() - t0) * 1000)

ms_per_step = np.mean(single_call_times)

print("Generation Time Scaling")
print("=" * 55)
print(f"  Single UNet call: {ms_per_step:.2f} ms")
print(f"\n{'Steps':>8} {'UNet Time':>12} {'+ Text Enc':>12} {'+ VAE Dec':>12} {'Total':>10}")
print("-" * 55)

text_enc_ms = 5.0  # Estimated
vae_dec_ms = 8.0   # Estimated

for steps in [1, 4, 8, 20, 50, 100]:
    unet_total = ms_per_step * steps * 2  # ×2 for CFG
    total = text_enc_ms + unet_total + vae_dec_ms
    print(f"{steps:>8} {unet_total:>10.0f}ms {text_enc_ms:>10.0f}ms {vae_dec_ms:>10.0f}ms {total:>8.0f}ms")

## 16. Practical Deployment Considerations

```
┌─────────────────────────────────────────────────────────────────┐
│        GENERATIVE AI DEPLOYMENT ARCHITECTURE                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                   │
│  Client Request                                                   │
│       │                                                           │
│       ▼                                                           │
│  ┌──────────────────────┐                                        │
│  │   API Gateway         │  Rate limiting, auth                   │
│  └──────────┬───────────┘                                        │
│             │                                                     │
│             ▼                                                     │
│  ┌──────────────────────┐                                        │
│  │   Request Queue       │  Async processing                     │
│  └──────────┬───────────┘                                        │
│             │                                                     │
│             ▼                                                     │
│  ┌──────────────────────┐                                        │
│  │   GPU Worker Pool     │  ONNX Runtime sessions               │
│  │   • Text Encoder      │  (shared across requests)             │
│  │   • UNet (FP16)       │                                       │
│  │   • VAE Decoder       │                                       │
│  └──────────┬───────────┘                                        │
│             │                                                     │
│             ▼                                                     │
│  ┌──────────────────────┐                                        │
│  │   Result Storage      │  S3/GCS for images                    │
│  └──────────────────────┘                                        │
│                                                                   │
│  Key Metrics:                                                     │
│  • Time to First Token (LLM): < 200ms                           │
│  • Image Generation: < 3s (20 steps, FP16, A100)                │
│  • GPU Utilization: > 80%                                        │
│  • Memory per request: ~4 GB (SD with FP16)                     │
└─────────────────────────────────────────────────────────────────┘
```

## 17. Summary

### Key Formulas

1. **Forward diffusion**: $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon$
2. **Reverse step**: $\mu_\theta = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta)$
3. **CFG**: $\tilde\epsilon = \epsilon_{uncond} + w(\epsilon_{cond} - \epsilon_{uncond})$
4. **KV-Cache**: $\text{Memory} = 2 \cdot L \cdot H \cdot d \cdot S \cdot B \cdot \text{bytes}$

### ONNX Export Strategy

- Export each pipeline component separately
- Use dynamic axes for batch dimension
- Keep iterative logic (diffusion loop, autoregressive loop) in Python
- Use FP16 for GPU, INT8 for CPU
- Profile UNet as it dominates inference time

In [ ]:
# Final summary of all components
print("""
╔═══════════════════════════════════════════════════════════════════╗
║         GENERATIVE AI + ONNX DEEP DIVE COMPLETE                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Diffusion Models:                                                ║
║  • Forward process adds noise: q(x_t|x_{t-1})                   ║
║  • UNet predicts noise: ε_θ(x_t, t, c)                          ║
║  • DDIM enables fewer steps (1000→20-50)                         ║
║  • CFG doubles UNet calls but improves quality                   ║
║  • Export: TextEncoder + UNet + VAE separately                   ║
║                                                                   ║
║  Autoregressive LLMs:                                             ║
║  • KV-Cache eliminates redundant computation                     ║
║  • Memory scales linearly with sequence length                   ║
║  • INT4/INT8 quantization for memory efficiency                  ║
║  • Speculative decoding for speed                                ║
║                                                                   ║
║  ONNX Best Practices:                                             ║
║  • Separate components for independent optimization              ║
║  • Keep loops in Python, tensor ops in ONNX                      ║
║  • FP16 on GPU, INT8 on CPU                                      ║
║  • Profile UNet/attention as bottleneck                          ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
""")